In [2]:
from langchain_groq import ChatGroq


In [4]:
llm = ChatGroq(
    temperature=0, 
    groq_api_key='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', 
    model_name="llama-3.3-70b-versatile"
)
response = llm.invoke("The first person to land on moon was ...")
print(response.content)

The first person to land on the moon was Neil Armstrong. He stepped out of the lunar module Eagle and onto the moon's surface on July 20, 1969, during the Apollo 11 mission. Armstrong famously declared, "That's one small step for man, one giant leap for mankind," as he became the first human to set foot on the moon.


In [11]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://careers.nike.com/senior-manager-engineering-global-converse-itc/job/R-82104")
page_data = loader.load().pop().page_content
print(page_data)





















Senior Manager Engineering, Global Converse, ITC












































Skip to main content
Open Virtual Assistant










Home


Career Areas


Total Rewards


Life@Nike


Purpose










Language





Select a Language

  Deutsch  
  English  
  Español (España)  
  Español (América Latina)  
  Français  
  Italiano  
  Nederlands  
  Polski  
  Tiếng Việt  
  Türkçe  
  简体中文  
  繁體中文  
  עִברִית  
  한국어  
  日本語  








Careers


















Close Menu







Careers






Chat






                                Home
                            



                                Career Areas
                            



                                Total Rewards
                            



                                Life@Nike
                            



                                Purpose
                            










Jordan Careers







Converse Careers










Language











Menu



Ret

In [13]:
from langchain_core.prompts import PromptTemplate

prompt_extract = PromptTemplate.from_template(
        """
        ### SCRAPED TEXT FROM WEBSITE:
        {page_data}
        ### INSTRUCTION:
        The scraped text is from the career's page of a website.
        Your job is to extract the job postings and return them in JSON format containing the 
        following keys: `role`, `experience`, `skills` and `description`.
        Only return the valid JSON.
        ### VALID JSON (NO PREAMBLE):    
        """
)

chain_extract = prompt_extract | llm 
res = chain_extract.invoke(input={'page_data':page_data})
type(res.content)

str

In [14]:
from langchain_core.output_parsers import JsonOutputParser

json_parser = JsonOutputParser()
json_res = json_parser.parse(res.content)
json_res

{'role': 'Senior Manager Engineering, Global Converse, ITC',
 'experience': '12+ years',
 'skills': ['BE/BTech - degree in Computer Science or equivalent',
  'MBA preferred',
  'Hands-on industry experience in developing data science models',
  'Backend & Frontend Development',
  'AI Agent Development',
  'DevOps, Deployment & Optimization',
  'Azure/AWS cloud native platforms',
  'SQL/NoSQL databases',
  'Git, Git Actions, Sonar (SonarQube)',
  'Automation platforms (such as RPA, iPaaS, GenAi, Agentic AI, MCP or similar)',
  'Python',
  'Modern data platforms like Snowflake and Databricks'],
 'description': 'Converse is seeking a Senior Manager Engineering, Global Converse, ITC to lead technical capabilities across technology Platforms, Supply chain and Marketplace domains. The successful candidate will have enthusiasm to enable AI orchestration and will work closely with internal teams across technology and business functions to deliver scalable platforms and consumer/enterprise appl

In [15]:
type(json_res)


dict

In [19]:
import pandas as pd

df = pd.read_csv("my_portfolio.csv")
df

,Techstack,Links
0,"React, Node.js, MongoDB",https://example.com/react-portfolio
1,"Angular,.NET, SQL Server",https://example.com/angular-portfolio
2,"Vue.js, Ruby on Rails, PostgreSQL",https://example.com/vue-portfolio
3,"Python, Django, MySQL",https://example.com/python-portfolio
4,"Java, Spring Boot, Oracle",https://example.com/java-portfolio
5,"Flutter, Firebase, GraphQL",https://example.com/flutter-portfolio
6,"WordPress, PHP, MySQL",https://example.com/wordpress-portfolio
7,"Magento, PHP, MySQL",https://example.com/magento-portfolio
8,"React Native, Node.js, MongoDB",https://example.com/react-native-portfolio
9,"iOS, Swift, Core Data",https://example.com/ios-portfolio


In [28]:
import uuid
import chromadb

client = chromadb.PersistentClient('vectorstore')
collection = client.get_or_create_collection(name="portfolio")

if not collection.count():
    for _, row in df.iterrows():
        collection.add(documents=row["Techstack"],
                       metadatas={"links": row["Links"]},
                       ids=[str(uuid.uuid4())])

In [30]:
links = collection.query(query_texts=['Experience in Python', 'Experience in React'], n_results=2).get('metadatas', [])
links

[[{'links': 'https://example.com/ml-python-portfolio'},
  {'links': 'https://example.com/python-portfolio'}],
 [{'links': 'https://example.com/react-portfolio'},
  {'links': 'https://example.com/react-native-portfolio'}]]

In [31]:
links = collection.query(query_texts=['Experience in AI', 'Experience in Python'], n_results=2).get('metadatas', [])
links

[[{'links': 'https://example.com/ml-python-portfolio'},
  {'links': 'https://example.com/ios-ar-portfolio'}],
 [{'links': 'https://example.com/ml-python-portfolio'},
  {'links': 'https://example.com/python-portfolio'}]]

In [35]:
job

{'role': 'Senior Manager Engineering, Global Converse, ITC',
 'experience': '12+ years',
 'skills': ['BE/BTech - degree in Computer Science or equivalent',
  'MBA preferred',
  'Hands-on industry experience in developing data science models',
  'Backend & Frontend Development',
  'AI Agent Development',
  'DevOps, Deployment & Optimization',
  'Azure/AWS cloud native platforms',
  'SQL/NoSQL databases',
  'Git, Git Actions, Sonar (SonarQube)',
  'Automation platforms (such as RPA, iPaaS, GenAi, Agentic AI, MCP or similar)',
  'Python',
  'Modern data platforms like Snowflake and Databricks'],
 'description': 'Converse is seeking a Senior Manager Engineering, Global Converse, ITC to lead technical capabilities across technology Platforms, Supply chain and Marketplace domains. The successful candidate will have enthusiasm to enable AI orchestration and will work closely with internal teams across technology and business functions to deliver scalable platforms and consumer/enterprise appl

In [36]:
job = json_res
job['skills']

['BE/BTech - degree in Computer Science or equivalent',
 'MBA preferred',
 'Hands-on industry experience in developing data science models',
 'Backend & Frontend Development',
 'AI Agent Development',
 'DevOps, Deployment & Optimization',
 'Azure/AWS cloud native platforms',
 'SQL/NoSQL databases',
 'Git, Git Actions, Sonar (SonarQube)',
 'Automation platforms (such as RPA, iPaaS, GenAi, Agentic AI, MCP or similar)',
 'Python',
 'Modern data platforms like Snowflake and Databricks']

In [37]:
prompt_email = PromptTemplate.from_template(
        """
        ### JOB DESCRIPTION:
        {job_description}
        
        ### INSTRUCTION:
        You are Mohan, a business development executive at AtliQ. AtliQ is an AI & Software Consulting company dedicated to facilitating
        the seamless integration of business processes through automated tools. 
        Over our experience, we have empowered numerous enterprises with tailored solutions, fostering scalability, 
        process optimization, cost reduction, and heightened overall efficiency. 
        Your job is to write a cold email to the client regarding the job mentioned above describing the capability of AtliQ 
        in fulfilling their needs.
        Also add the most relevant ones from the following links to showcase Atliq's portfolio: {link_list}
        Remember you are Mohan, BDE at AtliQ. 
        Do not provide a preamble.
        ### EMAIL (NO PREAMBLE):
        
        """
        )

chain_email = prompt_email | llm
res = chain_email.invoke({"job_description": str(job), "link_list": links})
print(res.content)

Subject: Expert Engineering Solutions for Converse's Technical Capabilities

Dear Hiring Manager,

I came across the job description for Senior Manager Engineering, Global Converse, ITC, and I'm excited to introduce AtliQ, an AI & Software Consulting company that can help Converse achieve its technical goals. With 12+ years of experience in developing data science models, backend & frontend development, AI agent development, and DevOps, our team is well-equipped to lead technical capabilities across technology platforms, supply chain, and marketplace domains.

At AtliQ, we have a strong portfolio in machine learning and Python development, which can be leveraged to enable AI orchestration and deliver scalable platforms and consumer/enterprise applications. Our expertise in Azure/AWS cloud native platforms, SQL/NoSQL databases, Git, Git Actions, Sonar (SonarQube), and automation platforms (such as RPA, iPaaS, GenAi, Agentic AI, MCP or similar) can help Converse optimize its technology s